In [ ]:
# !pip install bertopic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 6.9 MB/s eta 0:00:00


In [ ]:
# !git clone https://github.com/ranslemus/topic_modeling_KBMI4.git
# %cd topic_modeling_KBMI4
# !git switch gavriel-thesis

Cloning into 'topic_modeling_KBMI4'...
remote: Enumerating objects: 302, done.
remote: Counting objects: 100% (82/82), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 302 (delta 39), reused 51 (delta 19), pack-reused 220 (from 2)
Receiving objects: 100% (302/302), 57.66 MiB | 11.78 MiB/s, done.
Resolving deltas: 100% (129/129), done.
Filtering content: 100% (4/4), 294.40 MiB | 16.22 MiB/s, done.
/content/topic_modeling_KBMI4
Updating files: 100% (41/41), done.
Filtering content: 100% (9/9), 1.07 GiB | 15.14 MiB/s, done.
Branch 'gavriel-thesis' set up to track remote branch 'gavriel-thesis' from 'origin'.
Switched to a new branch 'gavriel-thesis'


In [1]:
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import plotly.express as px

from transformers import AutoTokenizer, AutoModel
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer
from tqdm.auto import tqdm
from cuml.manifold import UMAP
from cuml.cluster import HDBSCAN

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device :", device)

if device.type == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))

Device : cuda
GPU : NVIDIA GeForce RTX 3060


In [3]:
df = pd.read_csv("data/preprocessed_data.csv")

df.head()

,reviewId,bank,score,year,text
0,e17751da-bf2e-4a8f-a5a8-334206bb93ca,BCAMOBILE_REVIEWS,1,2025,ribet banget nih apk sumpah dikir verif dikit ...
1,3481f1d1-a22f-445f-ae2f-ed8c2c9dca53,BCAMOBILE_REVIEWS,2,2025,kenapa qris enggak bisa di pakai ya daritadi l...
2,af69ec6d-cc97-404e-b86a-e5f5b5f1f711,BCAMOBILE_REVIEWS,1,2025,aplikasi nya sampah kenapa tiba tiba keluar te...
3,32d28ac6-c538-4749-969f-8af060915d96,BCAMOBILE_REVIEWS,3,2025,bagus
4,f99259b7-0d45-4422-b667-6c4fd521638b,BCAMOBILE_REVIEWS,1,2025,tidak ada solusi ketika ada kendala di persuli...


In [4]:
documents = df["text"].astype(str).tolist()

print(f"Total documents : {len(documents):,}")

Total documents : 193,827


# IndoBERT

In [ ]:
MODEL_NAME = "indobenchmark/indobert-base-p1"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModel.from_pretrained(MODEL_NAME)

model.to(device)

model.eval()

e:\anaconda\envs\nlp\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\gavri\.cache\huggingface\hub\models--indobenchmark--indobert-base-p1. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP downloa

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(50000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


In [ ]:
def mean_pooling(model_output, attention_mask):

    token_embeddings = model_output.last_hidden_state

    input_mask_expanded = (
        attention_mask
        .unsqueeze(-1)
        .expand(token_embeddings.size())
        .float()
    )

    return torch.sum(
        token_embeddings * input_mask_expanded,
        dim=1
    ) / torch.clamp(
        input_mask_expanded.sum(dim=1),
        min=1e-9
    )

In [ ]:
def encode_documents(
    documents,
    batch_size=32,
    max_length=128
):

    embeddings = []

    with torch.no_grad():

        for i in tqdm(
            range(0, len(documents), batch_size)
        ):

            batch = documents[
                i:i+batch_size
            ]

            encoded_input = tokenizer(
                batch,
                padding=True,
                truncation=True,
                max_length=max_length,
                return_tensors="pt"
            )

            encoded_input = {
                k: v.to(device)
                for k, v in encoded_input.items()
            }

            model_output = model(**encoded_input)

            sentence_embeddings = mean_pooling(
                model_output,
                encoded_input["attention_mask"]
            )

            sentence_embeddings = (
                sentence_embeddings
                .cpu()
                .numpy()
            )

            embeddings.append(sentence_embeddings)

    return np.vstack(embeddings)

In [ ]:
embeddings = encode_documents(
    documents,
    batch_size=32,
    max_length=128
)

100%|██████████| 6058/6058 [26:05<00:00,  3.87it/s]


In [ ]:
print(embeddings.shape)

(193827, 768)


In [ ]:
embeddings[0]

array([ 1.32068610e+00,  3.98074418e-01, -6.73392862e-02,  6.81021631e-01,
       -1.96820974e-01,  1.08589363e+00, -8.56553018e-01,  2.23273388e-03,
        8.79841328e-01,  3.65444899e-01, -3.26138228e-01,  4.72277194e-01,
       -8.05487573e-01,  1.67667508e-01, -2.56380171e-01, -4.40765619e-01,
       -4.77702200e-01,  2.61944145e-01, -2.86765069e-01, -2.72093844e-02,
        7.49712348e-01,  1.74879134e-02,  2.77186837e-02, -1.01489611e-01,
       -5.50755799e-01, -4.60477501e-01,  3.93649340e-01,  8.93991292e-01,
       -5.85758351e-02, -1.70096517e-01, -2.30777428e-01,  5.49784362e-01,
        2.69129515e-01,  5.88295817e-01, -7.31480896e-01,  9.41245556e-01,
        4.00836855e-01,  1.14819241e+00, -5.05113244e-01, -9.31886211e-02,
       -1.09638429e+00,  7.68689096e-01, -1.85003400e-01,  1.06485210e-01,
       -8.93403411e-01,  6.15286827e-01,  5.15412211e-01,  8.41604114e-01,
       -5.12369573e-01, -3.84789519e-02, -7.70254806e-02,  3.15342516e-01,
        1.43759996e-01,  

In [ ]:
norms = np.linalg.norm(embeddings, axis=1)

print("Minimum Norm :", norms.min())
print("Maximum Norm :", norms.max())
print("Average Norm :", norms.mean())
print("Std Norm :", norms.std())

Minimum Norm : 12.87398
Maximum Norm : 26.276495
Average Norm : 18.317768
Std Norm : 2.1703527


In [ ]:
print("NaN :", np.isnan(embeddings).sum())
print("Inf :", np.isinf(embeddings).sum())

NaN : 0
Inf : 0


In [ ]:
np.save(
    "indobert_embeddings.npy",
    embeddings
)

# BERTopic

In [5]:
embeddings = np.load("indobert_embeddings.npy")

print("Embedding Shape :", embeddings.shape)

Embedding Shape : (193827, 768)


In [6]:
vectorizer_model = CountVectorizer(ngram_range=(1,2))

baseline UMAP for testing purpose

In [7]:
umap_model = UMAP(
    n_neighbors=30,
    n_components=5,
    metric="cosine",
    min_dist=0.0,
    random_state=42
)

baseline HDBSCAN

In [8]:
hdbscan_model = HDBSCAN(
    min_cluster_size=50,
    metric="euclidean",
    prediction_data=True
)

In [9]:
topic_model = BERTopic(
    embedding_model=None,
    calculate_probabilities=False,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    verbose=True
)

In [10]:
topics, probabilities = topic_model.fit_transform(
    documents,
    embeddings
)

2026-08-08 12:22:24,492 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-08-08 12:22:53,719 - BERTopic - Dimensionality - Completed ✓
2026-08-08 12:22:53,759 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-08-08 12:22:59,281 - BERTopic - Cluster - Completed ✓
2026-08-08 12:22:59,333 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-08-08 12:23:15,308 - BERTopic - Representation - Completed ✓


# Evaluation

Basic Statistics

In [11]:
topic_info = topic_model.get_topic_info()

topic_info.head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,133769,-1_enggak_bisa_di_aplikasi,"[enggak, bisa, di, aplikasi, sudah, enggak bis...",[ini bagaimana sih aplikasinya masak saya mau ...
1,0,12619,0_saldo_uang_saya_tapi,"[saldo, uang, saya, tapi, biaya, ada, potongan...",[enggak jelas saya top up gopay dari brimo tap...
2,1,3975,1_jaringan_sinyal_merah_bagus,"[jaringan, sinyal, merah, bagus, koneksi, inte...",[kenapa ini kok sinyal nya merah terus padahal...
3,2,3215,2_wajah_verifikasi wajah_verifikasi_gagal,"[wajah, verifikasi wajah, verifikasi, gagal, w...","[enggak bisa verifikasi wajah gagal terus, ver..."
4,3,3152,3_tolong_perbaiki_di perbaiki_tolong di,"[tolong, perbaiki, di perbaiki, tolong di, moh...",[tolong di perbaiki lagi setelah di update mal...
5,4,2257,4_malam_jam_00_tengah malam,"[malam, jam, 00, tengah malam, maintenance, te...","[tengah malam selalu gangguan, setiap jam 11 m..."
6,5,2031,5_kenapa_bisa di_di buka_buka,"[kenapa, bisa di, di buka, buka, kenapa aplika...","[kenapa aplikasi nya enggak bisa di buka, kena..."
7,6,1906,6_username_benar_password_sudah benar,"[username, benar, password, sudah benar, salah...",[enggak bisa login padahal username password s...
8,7,1502,7_mudah_membantu_sangat_sangat membantu,"[mudah, membantu, sangat, sangat membantu, cep...","[sangat mudah dan cepat, lebih mudah sangat me..."
9,8,1474,8_update malah_setelah_update_malah enggak,"[update malah, setelah, update, malah enggak, ...","[setelah update malah enggak bisa di buka, set..."


In [12]:
num_topics = len(topic_info) - 1

outlier_count = (np.array(topics) == -1).sum()

outlier_percentage = (
    outlier_count / len(topics)
) * 100

print(f"Topics              : {num_topics}")
print(f"Outliers            : {outlier_count:,}")
print(f"Outlier Percentage  : {outlier_percentage:.2f}%")

Topics              : 163
Outliers            : 133,769
Outlier Percentage  : 69.01%


Topic Size

In [13]:
topic_info[["Topic","Count"]]

,Topic,Count
0,-1,133769
1,0,12619
2,1,3975
3,2,3215
4,3,3152
...,...,...
159,158,53
160,159,52
161,160,52
162,161,51


Top Words

In [14]:
for topic in topic_info.Topic:

    if topic == -1:
        continue

    print("="*80)

    print(f"Topic {topic}")

    print(topic_model.get_topic(topic))

Topic 0
[('saldo', np.float64(0.012885139861161214)), ('uang', np.float64(0.00862615804984986)), ('saya', np.float64(0.00776881331742755)), ('tapi', np.float64(0.006674888247911256)), ('biaya', np.float64(0.006327588696965424)), ('ada', np.float64(0.0061325092924828396)), ('potongan', np.float64(0.005806711691391053)), ('tapi saldo', np.float64(0.005572194728400699)), ('terpotong', np.float64(0.00555558409267342)), ('transaksi', np.float64(0.005339044833841051))]
Topic 1
[('jaringan', np.float64(0.032923171104339066)), ('sinyal', np.float64(0.027512786743737833)), ('merah', np.float64(0.02487189056821121)), ('bagus', np.float64(0.022409005825047283)), ('koneksi', np.float64(0.02104662452409471)), ('internet', np.float64(0.019381328568905926)), ('indikator', np.float64(0.01909395956445264)), ('wifi', np.float64(0.018408687341723757)), ('padahal', np.float64(0.01790040602291725)), ('lampu', np.float64(0.017586002328606017))]
Topic 2
[('wajah', np.float64(0.07073576390186849)), ('verifika

Representative Reviews

In [15]:
representative_docs = topic_model.get_representative_docs()

for topic in representative_docs:

    if topic == -1:
        continue

    print("="*100)

    print(f"Topic {topic}")

    print()

    for doc in representative_docs[topic][:5]:

        print("-", doc)

    print()

Topic 0

- enggak jelas saya top up gopay dari brimo tapi enggak masuk tapi saldo brimo saya berkurang saya sudah menunggu 2x24 jam tapi saldo tetap enggak kembali kecewa banget
- saya tf 2 kali ke gopay saya tidak masuk tapi saldo kepotong saya komplain tapi respon nya sangat lambat dan sampai sekarang saldo nya enggak masuk ke akun gopay saya dan tidak balik lagi ke rekening bri saya pihak bri enggak ada tanggung jawab nya
- transfer ke bank lain saldo sudah berkurang tapi belum masuk ke rekening tujuan di cek mutasi sama bca sudah berhasil tapi tidak ada laporan berhasil di layar atau di inbox sama bca bagaimana ini

Topic 1

- kenapa ini kok sinyal nya merah terus padahal jaringan bagus
- enggak bisa masuk padahal jaringan bagus
- ini kenapa lampu indikator selalu merah padahal sinyal jaringan bagus

Topic 2

- enggak bisa verifikasi wajah gagal terus
- verifikasi wajah gagal terus
- verifikasi wajah gagal terus

Topic 3

- tolong di perbaiki lagi setelah di update malah force clos

silhoutte score

In [16]:
from sklearn.metrics import silhouette_score

mask = np.array(topics) != -1

silhouette = silhouette_score(
    topic_model.umap_model.embedding_[mask],
    np.array(topics)[mask]
)

print(f"Silhouette Score : {silhouette:.4f}")

Silhouette Score : 0.4062


In [17]:
from itertools import chain

top_n = 10
topic_words = []

for topic in topic_info.Topic:
    if topic == -1:
        continue

    words = [
        word
        for word, score in topic_model.get_topic(topic)[:top_n]
    ]
    topic_words.append(words)

unique_words = len(
    set(chain.from_iterable(topic_words))
)

total_words = len(topic_words) * top_n
topic_diversity = unique_words / total_words

print(f"Topic Diversity : {topic_diversity:.4f}")

Topic Diversity : 0.7344


Representative Reviews

In [18]:
representative_docs = topic_model.get_representative_docs()

for topic, docs in representative_docs.items():

    if topic == -1:
        continue

    print("="*100)

    print(f"TOPIC {topic}")

    print()

    for i, doc in enumerate(docs[:5],1):

        print(f"{i}. {doc}")

        print()

TOPIC 0

1. enggak jelas saya top up gopay dari brimo tapi enggak masuk tapi saldo brimo saya berkurang saya sudah menunggu 2x24 jam tapi saldo tetap enggak kembali kecewa banget

2. saya tf 2 kali ke gopay saya tidak masuk tapi saldo kepotong saya komplain tapi respon nya sangat lambat dan sampai sekarang saldo nya enggak masuk ke akun gopay saya dan tidak balik lagi ke rekening bri saya pihak bri enggak ada tanggung jawab nya

3. transfer ke bank lain saldo sudah berkurang tapi belum masuk ke rekening tujuan di cek mutasi sama bca sudah berhasil tapi tidak ada laporan berhasil di layar atau di inbox sama bca bagaimana ini

TOPIC 1

1. kenapa ini kok sinyal nya merah terus padahal jaringan bagus

2. enggak bisa masuk padahal jaringan bagus

3. ini kenapa lampu indikator selalu merah padahal sinyal jaringan bagus

TOPIC 2

1. enggak bisa verifikasi wajah gagal terus

2. verifikasi wajah gagal terus

3. verifikasi wajah gagal terus

TOPIC 3

1. tolong di perbaiki lagi setelah di update 

NPMI

In [19]:
analyzer = topic_model.vectorizer_model.build_analyzer()

In [20]:
doc.split()

['sangat', 'membantu']

In [21]:
tokenized_docs = [
    analyzer(doc)
    for doc in documents
]

In [ ]:
from gensim.corpora import Dictionary

dictionary = Dictionary(tokenized_docs)
topic_words = []

for topic in topic_info.Topic:

    if topic == -1:
        continue

    words = []

    for word, score in topic_model.get_topic(topic):
        if word in dictionary.token2id:
            words.append(word)
    # Need at least 2 words for coherence
    if len(words) >= 2:
        topic_words.append(words)

In [ ]:
# sanity check 
print(f"Valid Topics : {len(topic_words)}")

print()

print(topic_words[:3])

Valid Topics : 180

[['saldo', 'uang', 'saya', 'tapi', 'ada', 'biaya', 'ke', 'potongan', 'saldo saya', 'berkurang'], ['jaringan', 'sinyal', 'merah', 'bagus', 'koneksi', 'indikator', 'internet', 'lampu', 'wifi', 'merah terus'], ['wajah', 'verifikasi wajah', 'verifikasi', 'gagal', 'wajah gagal', 'gagal terus', 'selalu gagal', 'susah', 'wajah selalu', 'verivikasi']]


In [ ]:
from gensim.models.coherencemodel import CoherenceModel

coherence_model = CoherenceModel(
    topics=topic_words,
    texts=tokenized_docs,
    dictionary=dictionary,
    coherence="c_npmi"
)

npmi = coherence_model.get_coherence()
print(f"NPMI : {npmi:.4f}")

NPMI : 0.0691
